<a href="https://colab.research.google.com/github/igMoreira/claude-cert-notebooks/blob/main/build_with_claude_api/class_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
pip install Anthropic

In [54]:
from anthropic import Anthropic
from google.colab import userdata

MODEL = 'claude-haiku-4-5-20251001'
#MODEL = 'claude-sonnet-5'
MAX_TOKENS = 1000
API_KEY = userdata.get('API_KEY')
client = Anthropic(api_key=API_KEY)


In [87]:
import json

def generate_dataset():
    return """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "type": "Python" | "JSON" | "Regex"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

def user_message(messages, text):
  messages.append({'role': 'user', 'content': text})
  return messages

def assistant_message(messages, text):
  messages.append({'role': 'assistant', 'content': text})
  return messages

def chat(messages, system=None, stop_sequences=None):
  params = {
      'model':MODEL,
      'max_tokens':MAX_TOKENS,
      'messages':messages
  }
  if system:
    params['system'] = system
  if stop_sequences:
    params['stop_sequences'] = stop_sequences

  response = client.messages.create(**params)
  answer = response.content[0].text
  assistant_message(messages, answer)
  return answer

In [90]:
messages = []
user_message(messages, generate_dataset())
assistant_message(messages, "```json")
output = chat(messages, stop_sequences=['```'])
dataset = json.loads(output)
with open('dataset.json', 'w') as f:
  json.dump(dataset, f, indent=2)

In [93]:
import json, re, ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_by_code(testcase, output):
  if testcase['type'] == 'Python':
    score = validate_python(output)
  elif testcase['type'] == 'JSON':
    score = validate_json(output)
  elif testcase['type'] == 'Regex':
    score = validate_regex(output)
  return score

def grade_by_model(testcase, output):
  # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.

    Task: {testcase['task']}
    Solution: {output}

    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    messages = []
    user_message(messages, eval_prompt)
    assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

def run_prompt(testcase):
  prompt = f"""
  Solve the following task:

  {testcase["task"]}

  * Respond only with Python, JSON, or a plain Regex
  * Do not add any comments or commentary or explanation
  """
  messages = []
  user_message(messages, prompt)
  assistant_message(messages, "```code")
  answer = chat(messages, stop_sequences=['```'])
  return answer

def run_test_case(testcase):
  output = run_prompt(testcase)
  #TODO: grading
  model_grade = grade_by_model(testcase, output)
  model_score = model_grade['score']
  reasoning = model_grade['reasoning']
  code_score = grade_by_code(testcase, output)
  score = (model_score + code_score) / 2

  return {
      'score': score,
      'output': output,
      'testcase': testcase,
      'reasoning': reasoning
  }

def run_eval(dataset):
  results = []
  for data in dataset:
    results.append(run_test_case(data))
  return json.dumps(results, indent=2)

In [94]:
from statistics import mean

with open('dataset.json', 'r') as f:
  dataset = json.load(f)
results = json.loads(run_eval(dataset))
avg_score = mean([result['score'] for result in results])
print(f"Average score: {avg_score}")
print(results)

Average score: 8.0
[{'score': 8.25, 'output': "\nimport re\n\ndef parse_s3_bucket(s3_uri):\n    match = re.match(r's3://([^/]+)', s3_uri)\n    return match.group(1) if match else None\n", 'testcase': {'task': "Parse an AWS S3 bucket name from an S3 URI in the format 's3://bucket-name/key/path' and extract just the bucket name", 'type': 'Regex'}, 'reasoning': 'The solution correctly solves the core parsing task with a clean regex approach and defensive programming. However, it lacks robustness for production use. Adding basic input validation and documenting AWS bucket naming constraints would significantly improve reliability. The regex itself is sound but could be more explicit about what constitutes a valid bucket name.'}, {'score': 8.75, 'output': '\n{\n  "Version": "2012-10-17",\n  "Statement": [\n    {\n      "Effect": "Allow",\n      "Action": [\n        "s3:GetObject",\n        "s3:GetObjectVersion"\n      ],\n      "Resource": "arn:aws:s3:::bucket-name/*"\n    },\n    {\n      